# Fabric Workspace ↔ Deployment Pipeline Audit & Reclaim

Tenant-wide audit that maps every workspace to the deployment-pipeline stage(s) that pin it, then
(optionally) reclaims and deletes workspaces that can't be removed because a pipeline owns them.

Runs entirely against the **admin** REST APIs, so it sees the whole tenant without per-pipeline
membership. The identity running the notebook needs the **Fabric Administrator** role in Entra.

**Why this exists**: a workspace assigned to a deployment-pipeline stage can't be deleted until it's
unassigned, and unassigning normally needs *Pipeline Admin*. But a *Workspace Admin* can also unassign
(via the [Unassign Workspace](https://learn.microsoft.com/rest/api/power-bi/pipelines/unassign-workspace)
API), and a Fabric Admin can grant themselves Workspace Admin on any workspace
([AddUserAsAdmin](https://learn.microsoft.com/rest/api/power-bi/admin/groups-add-user-as-admin)). So the
full chain (grant self-admin -> unassign -> delete) is self-service - you never need the original
pipeline owner.

**Flow**
1. **Configure** - identity, scope, and the destructive-action gates (all OFF by default).
2. **Authenticate** - signed-in user by default; a Service Principal block is included but commented out.
3. **Discover** - enumerate workspaces and pipelines, then join them.
4. **Audit** - read-only report: which workspaces are pipeline-blocked, empty, orphaned, off-capacity.
5. **Reclaim** - gated. For an explicit list of workspace IDs: grant self-admin -> unassign -> delete.

> The discovery and audit cells are read-only. Nothing destructive runs unless you set
> `ENABLE_RECLAIM = True` **and** `DRY_RUN = False` in Configuration **and** populate
> `RECLAIM_WORKSPACE_IDS`.

## Configuration

Everything that controls scope and safety lives here. The reclaim gates are deliberately
conservative: with `DRY_RUN = True` and `ENABLE_RECLAIM = False`, a full run only ever *reports*.

In [ ]:
# ----- Identity --------------------------------------------------------------------
# "user" - run as the signed-in user (must hold the Fabric Administrator role). Default.
# "spn"  - service principal via Key Vault (see the commented block in Authentication).
#          Requires the tenant setting "Service principals can use admin APIs" plus the
#          SPN being granted rights for any write operation you enable.
AUTH_MODE = "user"

# Identity granted Workspace Admin during reclaim. Leave both as None to auto-resolve from
# the running identity's own token (UPN for a user, object ID for a service principal) - the
# Authentication cell fills them in. Only set these to grant a DIFFERENT principal, e.g. a
# security group ("<group-object-id>" + "Group").
SELF_PRINCIPAL = None
SELF_PRINCIPAL_TYPE = None   # auto: "User" for a person, "App" for an SPN; or set "Group"

# ----- Audit scope -----------------------------------------------------------------
# Count items per workspace via the Fabric admin API to flag empty workspaces.
# Adds one REST call per workspace - turn off for very large tenants if it's slow.
COUNT_ITEMS = True

# Flag workspaces not on a dedicated (Fabric/Premium) capacity as a cleanup signal.
FLAG_OFF_CAPACITY = True

# Exclude personal workspaces ("My workspace") - they can't be deleted here anyway.
EXCLUDE_PERSONAL = True

# ----- Reclaim gates (destructive - all OFF by default) ----------------------------
# Master switch. While False, the reclaim cell only prints what it *would* do.
ENABLE_RECLAIM = False

# Belt-and-braces. While True, every write (grant / unassign / delete) is logged but NOT
# sent, even if ENABLE_RECLAIM is True. Flip to False only once you've reviewed the audit.
DRY_RUN = True

# Reclaim runs in TWO phases, because a workspace-admin grant only takes effect on a fresh
# sign-in - Power BI propagates role changes "on next login", so granting and unassigning in
# the same session fails with 401 ALM_InvalidRequest_AccessToPipelineDenied.
#   "grant"   - grant yourself Workspace Admin on every target, verify, then STOP.
#               -> then restart the notebook session and wait a few minutes for propagation.
#   "execute" - re-acquire the token, unassign from all stages, and optionally delete.
# If you are ALREADY Pipeline Admin (or Workspace Admin) on the targets, skip straight to
# "execute" - the grant phase is only needed to bootstrap that access.
RECLAIM_PHASE = "grant"

# Self-grant Pipeline Admin via Admin - Pipelines UpdateUserAsAdmin. This is the RELIABLE
# unblock: Pipeline Admin is the role the unassign API actually wants. (The weaker
# "Workspace Admin can unassign" path is often denied with AccessToPipelineDenied.) Leave on.
GRANT_PIPELINE_ADMIN = True

# After unassigning from every stage, delete the workspace. Separate gate from unassign,
# so you can detach-only and review before deleting. Deleting the workspace needs Workspace
# Admin, which the grant phase also assigns.
DELETE_AFTER_UNASSIGN = False

# Explicit allow-list of workspace IDs to reclaim. Intentionally NOT auto-derived from the
# audit - paste in the GUIDs you've reviewed and decided to remove.
RECLAIM_WORKSPACE_IDS = [
    # "00000000-0000-0000-0000-000000000000",
]

## Authentication

By default this acquires a **Power BI** token (the `/admin`, `/pipelines`, and `/groups` endpoints all
live on `api.powerbi.com`) and a **Fabric** token (item counts) for the *signed-in user*. Because you
hold the Fabric Administrator role, the admin APIs and the write operations work with no extra tenant
setting and no SPN.

The Service Principal alternative is included but commented out. To use it: set `AUTH_MODE = "spn"`,
fill the Key Vault placeholders, and uncomment the block. The SPN path additionally requires the
admin-API tenant setting and that the SPN be permitted any write operation you enable.

In [ ]:
import base64
import json
import time
from typing import Any, Dict, List, Optional, Tuple

import requests

PBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

# Token audiences. The "pbi" alias also works for the Power BI resource; the explicit
# .default resource URL is used here to parallel the Fabric one.
PBI_RESOURCE = "https://analysis.windows.net/powerbi/api/.default"
FABRIC_RESOURCE = "https://api.fabric.microsoft.com/.default"


def _user_tokens() -> Tuple[str, str]:
    """Acquire (power_bi_token, fabric_token) for the signed-in user via the notebook runtime."""
    pbi = notebookutils.credentials.getToken(PBI_RESOURCE)
    fabric = notebookutils.credentials.getToken(FABRIC_RESOURCE)
    return pbi, fabric


# ----- Service Principal alternative (commented out) -------------------------------
# Set AUTH_MODE = "spn" and uncomment. Reads SPN creds from Key Vault and uses MSAL
# client-credentials to mint both tokens. The SPN needs admin-API access (tenant setting)
# and rights for any write operation you enable.
#
# import msal
#
# KEY_VAULT_URL = "https://<KeyVaultName>.vault.azure.net/"
# TENANT_ID_SECRET     = "<TenantIdSecretName>"
# CLIENT_ID_SECRET     = "<ServicePrincipalClientIdSecretName>"
# CLIENT_SECRET_SECRET = "<ServicePrincipalClientSecretSecretName>"
#
# def _spn_token(scope: str) -> str:
#     tenant_id = notebookutils.credentials.getSecret(KEY_VAULT_URL, TENANT_ID_SECRET)
#     client_id = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_ID_SECRET)
#     client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_SECRET)
#     app = msal.ConfidentialClientApplication(
#         client_id,
#         authority=f"https://login.microsoftonline.com/{tenant_id}",
#         client_credential=client_secret,
#     )
#     result = app.acquire_token_for_client(scopes=[scope])
#     if "access_token" not in result:
#         raise RuntimeError(f"SPN token request failed: {result.get('error_description', result)}")
#     return result["access_token"]
#
# def _spn_tokens() -> Tuple[str, str]:
#     return (
#         _spn_token("https://analysis.windows.net/powerbi/api/.default"),
#         _spn_token("https://api.fabric.microsoft.com/.default"),
#     )

if AUTH_MODE == "user":
    PBI_TOKEN, FABRIC_TOKEN = _user_tokens()
# elif AUTH_MODE == "spn":
#     PBI_TOKEN, FABRIC_TOKEN = _spn_tokens()
else:
    raise ValueError(f"Unsupported AUTH_MODE '{AUTH_MODE}' (expected 'user' or 'spn').")

PBI_HEADERS = {"Authorization": f"Bearer {PBI_TOKEN}", "Content-Type": "application/json"}
FABRIC_HEADERS = {"Authorization": f"Bearer {FABRIC_TOKEN}", "Content-Type": "application/json"}


def refresh_tokens() -> None:
    """Re-mint both tokens and rebuild the header dicts in place.

    Call this in the reclaim "execute" phase (after a self-admin grant + session restart)
    so the new Workspace Admin role is reflected. A genuine session restart is what forces
    Power BI to re-evaluate workspace permissions; this just picks up the fresh token.
    """
    global PBI_TOKEN, FABRIC_TOKEN, PBI_HEADERS, FABRIC_HEADERS
    if AUTH_MODE == "user":
        PBI_TOKEN, FABRIC_TOKEN = _user_tokens()
    # elif AUTH_MODE == "spn":
    #     PBI_TOKEN, FABRIC_TOKEN = _spn_tokens()
    PBI_HEADERS = {"Authorization": f"Bearer {PBI_TOKEN}", "Content-Type": "application/json"}
    FABRIC_HEADERS = {"Authorization": f"Bearer {FABRIC_TOKEN}", "Content-Type": "application/json"}


def _decode_jwt_claims(token: str) -> Dict[str, Any]:
    """Decode the (unverified) claims payload of a JWT - for identity inspection only."""
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)  # restore base64url padding
        return json.loads(base64.urlsafe_b64decode(payload))
    except (IndexError, ValueError):
        return {}


def resolve_self_principal() -> Tuple[str, str]:
    """Resolve (identifier, principalType) for the running identity from its token.

    Prefers the token's `upn` (a signed-in user) -> ("<upn>", "User"); falls back to `oid`
    (service principal, which has no upn) -> ("<objectId>", "App"); finally the runtime
    context. This targets the grant at exactly the identity making the API calls, and works
    for whoever runs the notebook - no hardcoded username.
    """
    claims = _decode_jwt_claims(PBI_TOKEN)
    upn = claims.get("upn") or claims.get("preferred_username") or claims.get("unique_name")
    if upn:
        return upn, "User"
    if claims.get("oid"):
        return claims["oid"], "App"
    ctx_local = notebookutils.runtime.context
    fallback = ctx_local.get("userName") or ctx_local.get("userId")
    if fallback:
        return fallback, "User"
    raise RuntimeError("Could not resolve the running identity from the token. "
                       "Set SELF_PRINCIPAL explicitly.")


# Resolve the running identity for the self-grant step, unless explicitly overridden.
ctx = notebookutils.runtime.context
if SELF_PRINCIPAL is None:
    SELF_PRINCIPAL, _detected_type = resolve_self_principal()
    if SELF_PRINCIPAL_TYPE is None:
        SELF_PRINCIPAL_TYPE = _detected_type
    print(f"Resolved running identity: {SELF_PRINCIPAL} ({SELF_PRINCIPAL_TYPE})")
else:
    SELF_PRINCIPAL_TYPE = SELF_PRINCIPAL_TYPE or "User"
    print(f"Using configured principal: {SELF_PRINCIPAL} ({SELF_PRINCIPAL_TYPE})")

print(f"Auth mode: {AUTH_MODE}  |  PBI token: {bool(PBI_TOKEN)}  |  Fabric token: {bool(FABRIC_TOKEN)}")

## Discovery

Three admin calls build the picture:
- **Workspaces** - `GET /admin/groups` (paged via `$skip`/`$top`): the full tenant inventory with
  capacity, state, and users.
- **Pipelines** - `GET /admin/pipelines?$expand=stages`: every deployment pipeline and the workspace
  pinned to each stage.
- **Items** (optional) - `GET /admin/workspaces/{id}/items` per workspace for an item count.

In [ ]:
def pbi_get_paged(path: str, params: Optional[Dict[str, Any]] = None,
                  page_size: int = 5000) -> List[Dict[str, Any]]:
    """GET a Power BI admin collection, paging via $skip/$top until exhausted.

    Args:
        path: Path under the Power BI base, e.g. "/admin/groups".
        params: Extra query params (e.g. {"$expand": "users"}).
        page_size: $top per request (admin groups caps at 5000).

    Returns:
        The concatenated "value" array across every page.
    """
    params = dict(params or {})
    results: List[Dict[str, Any]] = []
    skip = 0
    while True:
        params["$top"] = page_size
        params["$skip"] = skip
        resp = requests.get(f"{PBI_API_BASE}{path}", headers=PBI_HEADERS, params=params)
        if resp.status_code != 200:
            raise RuntimeError(f"GET {path} failed: {resp.status_code} {resp.text}")
        page = resp.json().get("value", [])
        results.extend(page)
        if len(page) < page_size:
            break
        skip += page_size
    return results


def get_pipelines_with_stages() -> List[Dict[str, Any]]:
    """List every deployment pipeline in the tenant with its stages expanded (admin scope)."""
    resp = requests.get(
        f"{PBI_API_BASE}/admin/pipelines",
        headers=PBI_HEADERS,
        params={"$expand": "stages"},
    )
    if resp.status_code != 200:
        raise RuntimeError(f"GET /admin/pipelines failed: {resp.status_code} {resp.text}")
    return resp.json().get("value", [])


def get_workspace_item_count(workspace_id: str) -> Optional[int]:
    """Return a workspace's item count via the Fabric admin API, or None if it can't be read."""
    resp = requests.get(
        f"{FABRIC_API_BASE}/admin/workspaces/{workspace_id}/items",
        headers=FABRIC_HEADERS,
    )
    if resp.status_code != 200:
        return None
    body = resp.json()
    return len(body.get("itemEntities", body.get("value", [])))


# ----- Run discovery ---------------------------------------------------------------
print("Listing workspaces (admin)...")
workspaces = pbi_get_paged("/admin/groups", params={"$expand": "users"})
print(f"  {len(workspaces)} workspaces")

print("Listing deployment pipelines (admin)...")
pipelines = get_pipelines_with_stages()
print(f"  {len(pipelines)} pipelines")

# Map workspace_id -> [ {pipelineId, pipelineName, stageOrder} ] for stages that pin a workspace.
ws_to_pipeline: Dict[str, List[Dict[str, Any]]] = {}
for pipe in pipelines:
    for stage in pipe.get("stages", []):
        ws_id = stage.get("workspaceId")
        if ws_id:
            ws_to_pipeline.setdefault(ws_id, []).append({
                "pipelineId": pipe["id"],
                "pipelineName": pipe.get("displayName") or pipe.get("name", ""),
                "stageOrder": stage.get("order"),
            })

ws_name_by_id = {ws["id"]: ws.get("name") for ws in workspaces}
print(f"  {len(ws_to_pipeline)} workspaces are pinned to a pipeline stage")

## Audit report (read-only)

One row per workspace: pipeline attachment, item count, capacity, state, and a `flags` column
summarising cleanup signals (`ON_PIPELINE`, `EMPTY`, `OFF_CAPACITY`, `NO_ADMIN`, plus any non-Active
state such as `ORPHANED`/`DELETED`). Nothing here mutates anything. `audit_df` is left in scope so you
can slice it further before choosing what to reclaim.

In [ ]:
import pandas as pd


def workspace_flags(ws: Dict[str, Any], pinned: List[Dict[str, Any]],
                    item_count: Optional[int]) -> List[str]:
    """Derive cleanup-signal flags for one workspace."""
    flags: List[str] = []
    if pinned:
        flags.append("ON_PIPELINE")
    if item_count == 0:
        flags.append("EMPTY")
    state = ws.get("state")
    if state and state != "Active":
        flags.append(state.upper())
    if FLAG_OFF_CAPACITY and not ws.get("isOnDedicatedCapacity", False):
        flags.append("OFF_CAPACITY")
    users = ws.get("users") or []
    if not any(u.get("groupUserAccessRight") == "Admin" for u in users):
        flags.append("NO_ADMIN")
    return flags


rows = []
for ws in workspaces:
    if EXCLUDE_PERSONAL and ws.get("type") == "PersonalGroup":
        continue
    ws_id = ws["id"]
    pinned = ws_to_pipeline.get(ws_id, [])
    item_count = get_workspace_item_count(ws_id) if COUNT_ITEMS else None
    rows.append({
        "workspaceId": ws_id,
        "name": ws.get("name"),
        "type": ws.get("type"),
        "state": ws.get("state"),
        "onCapacity": ws.get("isOnDedicatedCapacity", False),
        "capacityId": ws.get("capacityId"),
        "items": item_count,
        "pipelines": ", ".join(sorted({p["pipelineName"] for p in pinned})),
        "pipelineStages": len(pinned),
        "flags": ",".join(workspace_flags(ws, pinned, item_count)),
    })

audit_df = (
    pd.DataFrame(rows)
    .sort_values(by=["pipelineStages", "items", "name"],
                 ascending=[False, True, True], na_position="first")
    .reset_index(drop=True)
)

pinned_count = int(audit_df["pipelineStages"].gt(0).sum())
print(f"Audited {len(audit_df)} workspaces ({pinned_count} pinned to deployment pipelines).\n")

blocked = audit_df[audit_df["pipelineStages"] > 0]
print("----- Workspaces pinned to a deployment pipeline (delete-blocked) -----")
print(
    blocked[["name", "workspaceId", "pipelines", "items", "flags"]].to_string(index=False)
    if not blocked.empty else "  (none)"
)

candidates = audit_df[audit_df["flags"].str.contains("EMPTY|ORPHANED|NO_ADMIN|DELETED", na=False)]
print("\n----- Possible cleanup candidates (empty / orphaned) -----")
print(
    candidates[["name", "workspaceId", "items", "pipelines", "flags"]].to_string(index=False)
    if not candidates.empty else "  (none)"
)

# display() renders the full sortable grid in Fabric.
display(audit_df)

## Reclaim & delete (gated, destructive)

A workspace pinned to a pipeline stage can't be deleted until it's unassigned, and unassigning needs
**Pipeline Admin**. As a Fabric Admin you can grant yourself that directly with
[Admin - Pipelines UpdateUserAsAdmin](https://learn.microsoft.com/rest/api/power-bi/admin/pipelines-update-user-as-admin)
(`POST /admin/pipelines/{id}/users`) - the same API Microsoft documents for taking over an
[orphaned pipeline](https://learn.microsoft.com/fabric/cicd/troubleshoot-cicd#how-can-i-delete-a-pipeline-that-doesnt-have-an-owner-an-orphaned-pipeline).
The weaker "Workspace Admin can unassign" path often fails with `401 AccessToPipelineDenied`, so this
notebook self-grants **Pipeline Admin** as the primary unblock, and Workspace Admin too (needed to
delete the workspace afterwards).

New roles only take effect on the **next sign-in**, so reclaim runs in two phases with a session
restart between:

**Phase `grant`** (`RECLAIM_PHASE = "grant"`)
1. grant **Pipeline Admin** on each pipeline pinning a target - `POST /admin/pipelines/{id}/users`
2. grant **Workspace Admin** on each target workspace - `POST /admin/groups/{id}/users`
3. **stop** - then **restart the notebook session** (Run -> restart) and wait ~2-10 min for propagation.

**Phase `execute`** (`RECLAIM_PHASE = "execute"`, after the restart)
1. re-mint the token (`refresh_tokens()`)
2. **unassign** from every stage that pins the workspace -
   `POST /pipelines/{id}/stages/{order}/unassignWorkspace` (retries with token refresh + backoff)
3. *(optional)* **delete** the workspace - `DELETE /groups/{id}`

Guarded by `ENABLE_RECLAIM`, `DRY_RUN`, `RECLAIM_PHASE`, `GRANT_PIPELINE_ADMIN`, and
`DELETE_AFTER_UNASSIGN`. With the defaults (reclaim off, dry-run on) this cell only prints the plan.

> Pipeline-admin grants often propagate within a minute, so you can try `execute` in the *same*
> session first (the unassign retries with backoff); only restart if it still 401s. `UpdateUserAsAdmin`
> is capped at 200 requests/hour - the grant phase calls it once per unique pipeline.

In [ ]:
def grant_self_pipeline_admin(pipeline_id: str) -> None:
    """Grant SELF_PRINCIPAL Admin on a deployment pipeline (Admin - UpdateUserAsAdmin).

    This is the reliable unblock for unassign: Pipeline Admin is the role the unassign API
    expects. Requires the caller to be a Fabric Admin. Capped at 200 requests/hour.
    """
    body = {
        "identifier": SELF_PRINCIPAL,
        "accessRight": "Admin",
        "principalType": SELF_PRINCIPAL_TYPE,
    }
    resp = requests.post(
        f"{PBI_API_BASE}/admin/pipelines/{pipeline_id}/users",
        headers=PBI_HEADERS, json=body,
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(
            f"grant pipeline admin on {pipeline_id} failed: {resp.status_code} {resp.text}"
        )


def grant_self_workspace_admin(workspace_id: str) -> None:
    """Grant SELF_PRINCIPAL the Admin role on a workspace via the admin API."""
    body = {
        "identifier": SELF_PRINCIPAL,
        "principalType": SELF_PRINCIPAL_TYPE,
        "groupUserAccessRight": "Admin",
    }
    resp = requests.post(
        f"{PBI_API_BASE}/admin/groups/{workspace_id}/users",
        headers=PBI_HEADERS, json=body,
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"grant admin on {workspace_id} failed: {resp.status_code} {resp.text}")


def self_is_workspace_admin(workspace_id: str) -> bool:
    """Read back the workspace admins and check SELF_PRINCIPAL is among them (best-effort)."""
    resp = requests.get(f"{PBI_API_BASE}/admin/groups/{workspace_id}/users", headers=PBI_HEADERS)
    if resp.status_code != 200:
        return False
    target = (SELF_PRINCIPAL or "").lower()
    for u in resp.json().get("value", []):
        if u.get("groupUserAccessRight") != "Admin":
            continue
        ident = (u.get("identifier") or u.get("emailAddress") or "").lower()
        if ident == target or u.get("displayName", "").lower() == target:
            return True
    return False


def unassign_from_stage(pipeline_id: str, stage_order: int, attempts: int = 4) -> None:
    """Unassign whatever workspace occupies a pipeline stage.

    Retries on 401/403 with a token refresh and growing backoff, to ride out the residual
    propagation lag after a fresh sign-in. A persistent 401 means the role still isn't
    recognised - restart the session and wait longer, or you need Pipeline Admin.
    """
    last: Optional[requests.Response] = None
    for i in range(attempts):
        resp = requests.post(
            f"{PBI_API_BASE}/pipelines/{pipeline_id}/stages/{stage_order}/unassignWorkspace",
            headers=PBI_HEADERS,
        )
        if resp.status_code in (200, 201):
            return
        last = resp
        if resp.status_code in (401, 403) and i < attempts - 1:
            time.sleep(15 * (i + 1))
            refresh_tokens()
            continue
        break
    raise RuntimeError(
        f"unassign stage {stage_order} of pipeline {pipeline_id} failed: "
        f"{last.status_code} {last.text}"
    )


def delete_workspace(workspace_id: str) -> None:
    """Delete a workspace (caller must be Workspace Admin, e.g. via grant_self_workspace_admin)."""
    resp = requests.delete(f"{PBI_API_BASE}/groups/{workspace_id}", headers=PBI_HEADERS)
    if resp.status_code not in (200, 204):
        raise RuntimeError(f"delete {workspace_id} failed: {resp.status_code} {resp.text}")


def step(actions: List[str], label: str, fn) -> None:
    """Run an action unless DRY_RUN, appending a status line to `actions`."""
    if DRY_RUN:
        actions.append(f"DRYRUN  {label}")
        return
    fn()
    actions.append(f"DONE    {label}")


def grant_phase(workspace_id: str) -> Dict[str, Any]:
    """Grant Pipeline Admin on each pinning pipeline + Workspace Admin on the target."""
    name = ws_name_by_id.get(workspace_id, "<unknown>")
    pinned = ws_to_pipeline.get(workspace_id, [])
    actions: List[str] = []

    if GRANT_PIPELINE_ADMIN:
        for pid in sorted({p["pipelineId"] for p in pinned}):
            if pid in _granted_pipelines:
                actions.append(f"SKIP    pipeline admin (already granted this run) - {pid}")
                continue
            pname = next((p["pipelineName"] for p in pinned if p["pipelineId"] == pid), pid)
            step(actions, f"grant {SELF_PRINCIPAL} Pipeline Admin on '{pname}'",
                 lambda pid=pid: grant_self_pipeline_admin(pid))
            if not DRY_RUN:
                _granted_pipelines.add(pid)

    step(actions, f"grant {SELF_PRINCIPAL} Workspace Admin on '{name}'",
         lambda: grant_self_workspace_admin(workspace_id))
    if not DRY_RUN:
        ok = self_is_workspace_admin(workspace_id)
        actions.append(f"{'VERIFIED' if ok else 'PENDING '} workspace admin on '{name}'"
                       f"{'' if ok else ' (may still be propagating)'}")
    return {"workspaceId": workspace_id, "name": name, "actions": actions}


def execute_phase(workspace_id: str) -> Dict[str, Any]:
    """Unassign one workspace from every stage that pins it, then optionally delete."""
    name = ws_name_by_id.get(workspace_id, "<unknown>")
    pinned = ws_to_pipeline.get(workspace_id, [])
    actions: List[str] = []
    for p in pinned:
        step(actions, f"unassign from pipeline '{p['pipelineName']}' stage {p['stageOrder']}",
             lambda p=p: unassign_from_stage(p["pipelineId"], p["stageOrder"]))
    if DELETE_AFTER_UNASSIGN:
        step(actions, f"delete workspace '{name}'", lambda: delete_workspace(workspace_id))
    else:
        actions.append("SKIP    delete (DELETE_AFTER_UNASSIGN=False)")
    return {"workspaceId": workspace_id, "name": name,
            "pinnedStages": len(pinned), "actions": actions}


# ----- Execute (gated) -------------------------------------------------------------
_granted_pipelines: set = set()  # unique pipelines granted this run (UpdateUserAsAdmin is rate-limited)

if not ENABLE_RECLAIM:
    print("ENABLE_RECLAIM is False - reclaim disabled. Review the audit, then enable to proceed.")
elif not RECLAIM_WORKSPACE_IDS:
    print("RECLAIM_WORKSPACE_IDS is empty - nothing to reclaim.")
elif RECLAIM_PHASE not in ("grant", "execute"):
    raise ValueError(f"RECLAIM_PHASE must be 'grant' or 'execute', got '{RECLAIM_PHASE}'.")
else:
    unknown = [w for w in RECLAIM_WORKSPACE_IDS if w not in ws_name_by_id]
    if unknown:
        raise ValueError(f"These IDs are not in the audited tenant inventory: {unknown}")
    if not DRY_RUN and RECLAIM_PHASE == "grant" and not SELF_PRINCIPAL:
        raise ValueError("SELF_PRINCIPAL is unset - the Authentication cell should resolve it. "
                         "Re-run the auth cell, or set it explicitly.")

    mode = "DRY RUN (no writes sent)" if DRY_RUN else "LIVE (writes WILL be sent)"
    if RECLAIM_PHASE == "execute" and not DRY_RUN:
        refresh_tokens()  # pick up the role granted before the session restart

    print(f"Reclaim phase '{RECLAIM_PHASE}' on {len(RECLAIM_WORKSPACE_IDS)} workspace(s) - {mode}\n")
    runner = grant_phase if RECLAIM_PHASE == "grant" else execute_phase
    reclaim_results = [runner(w) for w in RECLAIM_WORKSPACE_IDS]
    for r in reclaim_results:
        print(f"\n{r['name']}  ({r['workspaceId']})")
        for a in r["actions"]:
            print(f"    {a}")

    if RECLAIM_PHASE == "grant" and not DRY_RUN:
        print("\nNEXT: restart the notebook session (so the new role takes effect), wait ~2-10 min,")
        print("      set RECLAIM_PHASE = \"execute\", and run all cells again.")
    print(f"\nDone. Phase '{RECLAIM_PHASE}' - {mode}.")